# Early Warning System - Final Integrated Version

**Purpose:** Quality-control second-look queue for atypical cases  
**Scope:** Discovery/review pathway, NOT diagnosis  
**Version:** Final with all improvements fully integrated

## Architecture

**Stage 0:** Define screening population (Mode A: NORMAL only | Mode B: NORMAL + uncertain)  
**Stage 1:** Score with embeddings (Mahalanobis distance - primary detection signal)  
**Stage 1B:** Enforce referral budget (exact top-k selection with ceil rounding)  
**Stage 2:** Explain with probability signals (NOT detection - explanation only)  
**Stage 3:** Discovery clustering (subgroup identification)

## ✅ Final Improvements Integrated

- **Normalized entropy** (0-1 scale, was 0-1.38)
- **Dynamic disease_mass** (not hard-coded column indices)
- **Suspicious NORMAL** (clearer than mid-band)
- **Overlap diagnostics** (normal-only, uncertain-only, both)
- **Error enrichment** (usefulness validation)
- **Review Priority** (not clinical risk)
- **Graceful clustering** (handles n<10)
- **Plain English summary** (accessible output)


In [1]:
# CELL 1: IMPORTS

import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from pathlib import Path
import json
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

print("✓ Imports successful")

✓ Imports successful


In [2]:
# CELL 2: CONFIGURATION

# ============================================================================
# PATHS
# ============================================================================

PROJECT_ROOT = Path(r"C:\Users\Ajant\Documents\Project MSc - CSC-40098\3 - Experiments")
EMBEDDINGS_ROOT = PROJECT_ROOT / "Embeddings"
RESULTS_DIR = PROJECT_ROOT / "Early_Warning_Detection" / "Results_Final_Integrated"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Embedding files
TRAIN_EMBEDDINGS = EMBEDDINGS_ROOT / "train_embeddings.npy"
TRAIN_LABELS = EMBEDDINGS_ROOT / "train_labels.npy"
VAL_EMBEDDINGS = EMBEDDINGS_ROOT / "val_embeddings.npy"
VAL_LABELS = EMBEDDINGS_ROOT / "val_labels.npy"
VAL_PREDICTIONS = EMBEDDINGS_ROOT / "val_predictions.npy"
VAL_PROBABILITIES = EMBEDDINGS_ROOT / "val_probabilities.npy"
TEST_EMBEDDINGS = EMBEDDINGS_ROOT / "test_embeddings.npy"
TEST_LABELS = EMBEDDINGS_ROOT / "test_labels.npy"
TEST_PREDICTIONS = EMBEDDINGS_ROOT / "test_predictions.npy"
TEST_PROBABILITIES = EMBEDDINGS_ROOT / "test_probabilities.npy"

# ============================================================================
# STAGE 0: SCREENING MODE
# ============================================================================

SCREENING_MODE = "mode_b"  # Options: "mode_a" (NORMAL only), "mode_b" (NORMAL + uncertain)

# Mode B thresholds (UPDATED WITH FINAL IMPROVEMENTS)
MARGIN_THRESHOLD = 0.3      # Low margin: top1 - top2 < 0.3
ENTROPY_THRESHOLD = 0.7     # High entropy > 0.5
DISEASE_MASS_THRESHOLD = 0.2  # Suspicious NORMAL: disease_mass > 0.2

# ============================================================================
# STAGE 1: DETECTION PARAMETERS
# ============================================================================

DETECTION_METHOD = "mahalanobis"  # Primary: embeddings-based anomaly detection
REFERRAL_BUDGET = 0.05            # 5% of screened population
BUDGET_ROUNDING = "ceil"          # Options: "ceil" (conservative), "round" (nearest)

# ============================================================================
# STAGE 2: EXPLANATION (PRIORITY LEVELS - NOT RISK)
# ============================================================================

N_PRIORITY_LEVELS = 4  # Review Priority 1 (highest) to Priority 4 (lowest)

# ============================================================================
# STAGE 3: CLUSTERING
# ============================================================================

CLUSTER_MIN_K = 2
CLUSTER_MAX_K = 6
CLUSTER_MIN_SAMPLES = 10  # Minimum samples required for clustering

# ============================================================================
# SENSITIVITY ANALYSIS
# ============================================================================

RUN_SENSITIVITY_ANALYSIS = True  # Set False to skip (saves time)

# Test these threshold combinations (for Stage 0 Mode B)

SENSITIVITY_CONFIGS = [
    {"name": "Conservative", "margin": 0.2, "entropy": 0.75, "dm_thresh": 0.25},
    {"name": "Default", "margin": 0.3, "entropy": 0.7, "dm_thresh": 0.2},
    {"name": "Liberal", "margin": 0.4, "entropy": 0.65, "dm_thresh": 0.15},
]

# ============================================================================
# MULTI-MODEL DISAGREEMENT (PLACEHOLDER)
# ============================================================================

ENABLE_DISAGREEMENT = False  # Enable when multiple model predictions available
DISAGREEMENT_PREDICTIONS = []  # List of prediction arrays from different models

# ============================================================================
# GENERAL
# ============================================================================

CLASS_NAMES = ['CNV', 'DME', 'DRUSEN', 'NORMAL']
NORMAL_CLASS_IDX = 3
RANDOM_STATE = 42

np.random.seed(RANDOM_STATE)

print("="*80)
print("EARLY WARNING SYSTEM - REFINED CONFIGURATION")
print("="*80)
print(f"Screening mode: {SCREENING_MODE}")
print(f"Detection method: {DETECTION_METHOD} (embeddings-based)")
print(f"Referral budget: {REFERRAL_BUDGET:.1%} (rounding: {BUDGET_ROUNDING})")
print(f"Priority levels: {N_PRIORITY_LEVELS} (explanatory, not clinical risk)")
print(f"Sensitivity analysis: {'Enabled' if RUN_SENSITIVITY_ANALYSIS else 'Disabled'}")
print(f"Results directory: {RESULTS_DIR.name}")
print("="*80)

EARLY WARNING SYSTEM - REFINED CONFIGURATION
Screening mode: mode_b
Detection method: mahalanobis (embeddings-based)
Referral budget: 5.0% (rounding: ceil)
Priority levels: 4 (explanatory, not clinical risk)
Sensitivity analysis: Enabled
Results directory: Results_Final_Integrated


In [3]:
# CELL 3: UTILITY FUNCTIONS (WITH ALL FINAL IMPROVEMENTS)

def compute_screening_mask(predictions, probabilities, mode="mode_a", 
                          margin_thresh=0.3, entropy_thresh=0.7, 
                          disease_mass_thresh=0.2, normal_idx=3):
    """
    Compute screening population mask with corrected logic.
    
    Mode A: Only predicted NORMAL (conservative safety net)
    Mode B: Predicted NORMAL OR uncertain cases
    
    IMPROVEMENTS:
    - Normalized entropy to [0,1] scale (was [0, ln(4)])
    - Dynamic disease_mass (not hard-coded to columns 0-2)
    - Suspicious NORMAL = high disease_mass among NORMAL predictions
    - Overlap diagnostics for interpretability
    
    Returns:
        mask: boolean array (cases to screen)
        components: dict with detailed breakdown including overlaps
    """
    n_samples = len(predictions)
    n_classes = probabilities.shape[1]
    
    # Component A: Predicted NORMAL
    pred_normal = (predictions == normal_idx)
    
    if mode == "mode_a":
        # Mode A: NORMAL predictions only
        mask = pred_normal
        components = {
            "predicted_normal": int(pred_normal.sum()),
            "screened_total": int(mask.sum())
        }
        
    elif mode == "mode_b":
        # Compute uncertainty signals
        
        # 1. Margin (confidence gap)
        sorted_probs = np.sort(probabilities, axis=1)[:, ::-1]
        margin = sorted_probs[:, 0] - sorted_probs[:, 1]
        low_margin = (margin < margin_thresh)
        
        # 2. Normalized entropy (FIXED: now 0-1 scale)
        epsilon = 1e-10
        entropy_raw = -np.sum(probabilities * np.log(probabilities + epsilon), axis=1)
        entropy_norm = entropy_raw / np.log(n_classes)  # Normalize to [0,1]
        high_entropy = (entropy_norm > entropy_thresh)
        
        # 3. Disease mass (FIXED: dynamic indices)
        disease_indices = [i for i in range(n_classes) if i != normal_idx]
        disease_mass = probabilities[:, disease_indices].sum(axis=1)
        
        # Component B: Suspicious NORMAL (predicted NORMAL but high disease_mass)
        # This catches NORMALs where model assigned significant disease probability
        suspicious_normal = pred_normal & (disease_mass > disease_mass_thresh)
        
        # Component C: Uncertain (low margin OR high entropy)
        uncertain = low_margin | high_entropy
        
        # Mode B mask: NORMAL OR uncertain (union logic)
        mask = pred_normal | uncertain
        
        # Overlap diagnostics for interpretability
        normal_only = pred_normal & ~uncertain
        uncertain_only = ~pred_normal & uncertain
        both = pred_normal & uncertain
        
        components = {
            # Primary components
            "predicted_normal": int(pred_normal.sum()),
            "low_margin": int(low_margin.sum()),
            "high_entropy": int(high_entropy.sum()),
            "suspicious_normal": int(suspicious_normal.sum()),
            
            # Derived
            "uncertain_total": int(uncertain.sum()),
            
            # Overlap analysis (NEW - critical for understanding)
            "normal_only": int(normal_only.sum()),
            "uncertain_only": int(uncertain_only.sum()),
            "both_normal_and_uncertain": int(both.sum()),
            
            # Final
            "screened_total": int(mask.sum())
        }
    
    else:
        raise ValueError(f"Unknown screening mode: {mode}")
    
    return mask, components


def print_screening_breakdown(components, mode, total_samples):
    """Print comprehensive screening breakdown with overlap analysis."""
    
    print(f"\nScreening mode: {mode.upper()}")
    print(f"Total samples: {total_samples:,}")
    print(f"\n{'='*60}")
    print("COMPONENT BREAKDOWN")
    print('='*60)
    
    if mode == "mode_a":
        print(f"  • Predicted NORMAL: {components['predicted_normal']:,}")
        print(f"\n  → Screened: {components['screened_total']:,} ({components['screened_total']/total_samples*100:.1f}% of total)")
        
    elif mode == "mode_b":
        print(f"Primary components:")
        print(f"  • Predicted NORMAL: {components['predicted_normal']:,}")
        print(f"  • Low margin (low confidence): {components['low_margin']:,}")
        print(f"  • High entropy (high uncertainty): {components['high_entropy']:,}")
        print(f"  • Suspicious NORMAL (high disease_mass): {components['suspicious_normal']:,}")
        
        print(f"\n{'-'*60}")
        print(f"Uncertainty analysis:")
        print(f"  • Total uncertain (margin OR entropy): {components['uncertain_total']:,}")
        
        print(f"\n{'-'*60}")
        print(f"Overlap analysis (how components combine):")
        print(f"  • NORMAL only (not uncertain): {components['normal_only']:,}")
        print(f"  • Uncertain only (not NORMAL pred): {components['uncertain_only']:,}")
        print(f"  • Both NORMAL and uncertain: {components['both_normal_and_uncertain']:,}")
        
        print(f"\n{'='*60}")
        print(f"→ SCREENED (union): {components['screened_total']:,} ({components['screened_total']/total_samples*100:.1f}% of total)")
        print('='*60)
        
        # Interpretation guidance
        pct_uncertain_only = components['uncertain_only'] / components['screened_total'] * 100
        pct_normal_only = components['normal_only'] / components['screened_total'] * 100
        
        print(f"\nInterpretation:")
        if pct_uncertain_only > 50:
            print(f"  ℹ️  Mode B primarily adding uncertain disease predictions ({pct_uncertain_only:.0f}%)")
            print(f"      System casts a wide net beyond just NORMAL predictions")
        elif pct_normal_only > 70:
            print(f"  ℹ️  Mode B primarily screening NORMAL predictions ({pct_normal_only:.0f}%)")
            print(f"      Uncertainty criteria add relatively few cases")
        else:
            print(f"  ℹ️  Balanced mix: {pct_normal_only:.0f}% NORMAL-only, {pct_uncertain_only:.0f}% uncertain-only")


def apply_referral_budget(scores, budget=0.05, rounding="ceil"):
    """
    Apply referral budget using top-k selection.
    
    Args:
        scores: anomaly scores (higher = more anomalous)
        budget: fraction to flag (0.0 to 1.0)
        rounding: "ceil" (conservative) or "round" (nearest)
    
    Returns:
        flagged_indices: indices of flagged cases
        n_flagged: number flagged
        actual_rate: actual flagging rate achieved
    """
    n_total = len(scores)
    
    if rounding == "ceil":
        n_to_flag = int(np.ceil(n_total * budget))
    elif rounding == "round":
        n_to_flag = int(np.round(n_total * budget))
    else:
        raise ValueError(f"Unknown rounding: {rounding}")
    
    # Ensure at least 1 if budget > 0
    if budget > 0 and n_to_flag == 0:
        n_to_flag = 1
    
    # Top-k selection (higher scores first)
    sorted_indices = np.argsort(scores)[::-1]
    flagged_indices = sorted_indices[:n_to_flag]
    
    actual_rate = n_to_flag / n_total
    
    return flagged_indices, n_to_flag, actual_rate


print("✓ Utility functions defined (with all final improvements)")


✓ Utility functions defined (with all final improvements)


In [4]:
# CELL 4: LOAD TRAIN NORMAL - COMPUTE REFERENCE STATISTICS

print("\n" + "="*80)
print("LOADING TRAIN NORMAL - COMPUTING REFERENCE STATISTICS")
print("="*80)

# Load train data
train_embeddings = np.load(TRAIN_EMBEDDINGS)
train_labels = np.load(TRAIN_LABELS)

# Filter NORMAL class (use labels, not predictions - this is ground truth)
train_normal_mask = (train_labels == NORMAL_CLASS_IDX)
train_normal_embeddings = train_embeddings[train_normal_mask]

print(f"Train NORMAL samples (ground truth): {len(train_normal_embeddings):,}")

# Compute reference statistics for Mahalanobis distance
print("\nComputing reference statistics (mean, covariance)...")
normal_mean = train_normal_embeddings.mean(axis=0)
normal_cov = np.cov(train_normal_embeddings.T)

# Regularize for numerical stability
reg_factor = 1e-6
normal_cov_reg = normal_cov + np.eye(normal_cov.shape[0]) * reg_factor
normal_cov_inv = np.linalg.inv(normal_cov_reg)

print(f"✓ Mean shape: {normal_mean.shape}")
print(f"✓ Covariance shape: {normal_cov.shape}")
print(f"✓ Regularization: {reg_factor}")

def compute_mahalanobis(embedding):
    """Compute Mahalanobis distance to NORMAL centroid (higher = more anomalous)."""
    diff = embedding - normal_mean
    return np.sqrt(diff @ normal_cov_inv @ diff.T)

print("\n✓ Reference statistics computed")
print("✓ Mahalanobis distance function ready")
print("="*80)


LOADING TRAIN NORMAL - COMPUTING REFERENCE STATISTICS
Train NORMAL samples (ground truth): 19,275

Computing reference statistics (mean, covariance)...
✓ Mean shape: (2048,)
✓ Covariance shape: (2048, 2048)
✓ Regularization: 1e-06

✓ Reference statistics computed
✓ Mahalanobis distance function ready


In [5]:
# CELL 5: SENSITIVITY ANALYSIS (OPTIONAL - STAGE 0 THRESHOLDS)

if RUN_SENSITIVITY_ANALYSIS and SCREENING_MODE == "mode_b":
    print("\n" + "="*80)
    print("SENSITIVITY ANALYSIS - STAGE 0 THRESHOLD IMPACT")
    print("="*80)
    print("Testing how screening thresholds affect population size and flagging...\n")
    
    # Load TEST data for sensitivity test
    test_embeddings_sens = np.load(TEST_EMBEDDINGS)
    test_predictions_sens = np.load(TEST_PREDICTIONS)
    test_probabilities_sens = np.load(TEST_PROBABILITIES)
    
    sensitivity_results = []
    
    for config in SENSITIVITY_CONFIGS:
        # Apply screening with this config
        mask, components = compute_screening_mask(
            test_predictions_sens,
            test_probabilities_sens,
            mode="mode_b",
            margin_thresh=config["margin"],
            entropy_thresh=config["entropy"],
            disease_mass_thresh=config["dm_thresh"],  # FIXED: was dm_min/dm_max
            normal_idx=NORMAL_CLASS_IDX
        )
        
        screened_embeddings = test_embeddings_sens[mask]
        
        # Compute scores and apply budget
        scores = np.array([compute_mahalanobis(emb) for emb in screened_embeddings])
        flagged_idx, n_flagged, actual_rate = apply_referral_budget(
            scores, REFERRAL_BUDGET, BUDGET_ROUNDING
        )
        
        sensitivity_results.append({
            "Configuration": config["name"],
            "Screened": len(screened_embeddings),
            "Screened_%": f"{len(screened_embeddings) / len(test_embeddings_sens) * 100:.1f}",
            "Flagged": n_flagged,
            "Flagged_Rate_%": f"{actual_rate * 100:.1f}"
        })
    
    # Display results table
    sens_df = pd.DataFrame(sensitivity_results)
    print("Sensitivity Analysis Results:")
    print("(Testing on TEST set with different Mode B thresholds)\n")
    print(sens_df.to_string(index=False))
    print("\n✓ Sensitivity analysis complete")
    print("="*80)
else:
    sensitivity_results = None
    if not RUN_SENSITIVITY_ANALYSIS:
        print("\n⊘ Sensitivity analysis skipped (RUN_SENSITIVITY_ANALYSIS=False)")
    elif SCREENING_MODE == "mode_a":
        print("\n⊘ Sensitivity analysis not applicable to Mode A")


SENSITIVITY ANALYSIS - STAGE 0 THRESHOLD IMPACT
Testing how screening thresholds affect population size and flagging...

Sensitivity Analysis Results:
(Testing on TEST set with different Mode B thresholds)

Configuration  Screened Screened_%  Flagged Flagged_Rate_%
 Conservative       247       25.5       13            5.3
      Default       250       25.8       13            5.2
      Liberal       253       26.1       13            5.1

✓ Sensitivity analysis complete


In [6]:
# CELL 6: STAGE 0 - DEFINE SCREENING POPULATION (VAL)

print("\n" + "="*80)
print("STAGE 0: DEFINING SCREENING POPULATION (VAL)")
print("="*80)

# Load VAL data
val_embeddings = np.load(VAL_EMBEDDINGS)
val_labels = np.load(VAL_LABELS)
val_predictions = np.load(VAL_PREDICTIONS)
val_probabilities = np.load(VAL_PROBABILITIES)

print(f"Total VAL samples: {len(val_embeddings):,}")

# Apply improved screening logic
val_screen_mask, val_components = compute_screening_mask(
    val_predictions,
    val_probabilities,
    mode=SCREENING_MODE,
    margin_thresh=MARGIN_THRESHOLD,
    entropy_thresh=ENTROPY_THRESHOLD,
    disease_mass_thresh=DISEASE_MASS_THRESHOLD,
    normal_idx=NORMAL_CLASS_IDX
)

# Print detailed breakdown with overlap analysis
print_screening_breakdown(val_components, SCREENING_MODE, len(val_embeddings))

# Extract screened population
val_screen_embeddings = val_embeddings[val_screen_mask]
val_screen_probabilities = val_probabilities[val_screen_mask]
val_screen_predictions = val_predictions[val_screen_mask]
val_screen_labels = val_labels[val_screen_mask]

print("\n✓ VAL screening population defined")
print("="*80)


STAGE 0: DEFINING SCREENING POPULATION (VAL)
Total VAL samples: 8,369

Screening mode: MODE_B
Total samples: 8,369

COMPONENT BREAKDOWN
Primary components:
  • Predicted NORMAL: 3,372
  • Low margin (low confidence): 129
  • High entropy (high uncertainty): 11
  • Suspicious NORMAL (high disease_mass): 84

------------------------------------------------------------
Uncertainty analysis:
  • Total uncertain (margin OR entropy): 131

------------------------------------------------------------
Overlap analysis (how components combine):
  • NORMAL only (not uncertain): 3,350
  • Uncertain only (not NORMAL pred): 109
  • Both NORMAL and uncertain: 22

→ SCREENED (union): 3,481 (41.6% of total)

Interpretation:
  ℹ️  Mode B primarily screening NORMAL predictions (96%)
      Uncertainty criteria add relatively few cases

✓ VAL screening population defined


In [7]:
# CELL 7: STAGE 1 - ANOMALY SCORING (VAL)

print("\n" + "="*80)
print("STAGE 1: ANOMALY SCORING (VAL) - EMBEDDINGS-BASED DETECTION")
print("="*80)

print(f"Computing {DETECTION_METHOD} scores for {len(val_screen_embeddings):,} candidates...")
print("(Primary detection signal: embeddings, NOT probabilities)\n")

val_anomaly_scores = np.array([
    compute_mahalanobis(emb) for emb in val_screen_embeddings
])

print(f"✓ Anomaly scores computed")
print(f"  Min: {val_anomaly_scores.min():.4f}")
print(f"  Max: {val_anomaly_scores.max():.4f}")
print(f"  Mean: {val_anomaly_scores.mean():.4f}")
print(f"  Median: {np.median(val_anomaly_scores):.4f}")
print(f"  Std: {val_anomaly_scores.std():.4f}")
print("="*80)


STAGE 1: ANOMALY SCORING (VAL) - EMBEDDINGS-BASED DETECTION
Computing mahalanobis scores for 3,481 candidates...
(Primary detection signal: embeddings, NOT probabilities)

✓ Anomaly scores computed
  Min: 6.1240
  Max: 65.6475
  Mean: 10.7150
  Median: 9.6043
  Std: 5.0349


In [8]:
# CELL 8: STAGE 1B - ENFORCE REFERRAL BUDGET (VAL)

print("\n" + "="*80)
print("STAGE 1B: ENFORCE REFERRAL BUDGET (VAL) - EXACT TOP-K SELECTION")
print("="*80)

val_flagged_indices, val_n_flagged, val_actual_rate = apply_referral_budget(
    val_anomaly_scores,
    budget=REFERRAL_BUDGET,
    rounding=BUDGET_ROUNDING
)

val_not_flagged_indices = np.array([i for i in range(len(val_screen_embeddings)) 
                                     if i not in val_flagged_indices])

val_threshold = val_anomaly_scores[val_flagged_indices[-1]]

print(f"Referral budget: {REFERRAL_BUDGET:.1%}")
print(f"Rounding method: {BUDGET_ROUNDING}")
print(f"Screened population: {len(val_screen_embeddings):,}")
print(f"\nBudget application:")
print(f"  Target: {REFERRAL_BUDGET * len(val_screen_embeddings):.1f} cases")
print(f"  Flagged: {val_n_flagged} cases")
print(f"  Actual rate: {val_actual_rate:.2%} (difference: {abs(val_actual_rate - REFERRAL_BUDGET):.2%})")
print(f"\n✓ Top-{val_n_flagged} selected (exact budget enforcement)")
print(f"  Effective threshold: {val_threshold:.4f} (informational only - not reused)")
print(f"  Score range (flagged): [{val_anomaly_scores[val_flagged_indices].min():.4f}, {val_anomaly_scores[val_flagged_indices].max():.4f}]")
print(f"  Score range (not flagged): [{val_anomaly_scores[val_not_flagged_indices].min():.4f}, {val_anomaly_scores[val_not_flagged_indices].max():.4f}]")
print("="*80)


STAGE 1B: ENFORCE REFERRAL BUDGET (VAL) - EXACT TOP-K SELECTION
Referral budget: 5.0%
Rounding method: ceil
Screened population: 3,481

Budget application:
  Target: 174.1 cases
  Flagged: 175 cases
  Actual rate: 5.03% (difference: 0.03%)

✓ Top-175 selected (exact budget enforcement)
  Effective threshold: 15.7198 (informational only - not reused)
  Score range (flagged): [15.7198, 65.6475]
  Score range (not flagged): [6.1240, 15.6557]


In [9]:
# CELL 9: STAGE 2 - EXTRACT EXPLANATION SIGNALS (VAL) - NOT DETECTION

print("\n" + "="*80)
print("STAGE 2: EXTRACT EXPLANATION SIGNALS (VAL FLAGGED)")
print("="*80)
print("Probability signals used for EXPLANATION only, NOT primary detection\n")

# Extract signals for flagged cases
val_flagged_probs = val_screen_probabilities[val_flagged_indices]

# Compute explanation signals
sorted_probs = np.sort(val_flagged_probs, axis=1)[:, ::-1]
val_flagged_margin = sorted_probs[:, 0] - sorted_probs[:, 1]

epsilon = 1e-10
val_flagged_entropy = -np.sum(val_flagged_probs * np.log(val_flagged_probs + epsilon), axis=1)

val_flagged_disease_mass = val_flagged_probs[:, 0] + val_flagged_probs[:, 1] + val_flagged_probs[:, 2]

print(f"✓ Signals extracted for {len(val_flagged_indices):,} flagged cases")
print(f"\nExplanation signal statistics:")
print(f"  Margin (confidence): Mean={val_flagged_margin.mean():.3f}, Std={val_flagged_margin.std():.3f}")
print(f"  Entropy (uncertainty): Mean={val_flagged_entropy.mean():.3f}, Std={val_flagged_entropy.std():.3f}")
print(f"  Disease mass: Mean={val_flagged_disease_mass.mean():.3f}, Std={val_flagged_disease_mass.std():.3f}")
print("\nNote: These signals explain WHY a case might be atypical, but did NOT drive the flagging decision.")
print("="*80)


STAGE 2: EXTRACT EXPLANATION SIGNALS (VAL FLAGGED)
Probability signals used for EXPLANATION only, NOT primary detection

✓ Signals extracted for 175 flagged cases

Explanation signal statistics:
  Margin (confidence): Mean=0.387, Std=0.333
  Entropy (uncertainty): Mean=0.593, Std=0.266
  Disease mass: Mean=0.591, Std=0.408

Note: These signals explain WHY a case might be atypical, but did NOT drive the flagging decision.


In [10]:
# CELL 10: STAGE 2B - ASSIGN REVIEW PRIORITY LEVELS (VAL) - NOT CLINICAL RISK

print("\n" + "="*80)
print("STAGE 2B: ASSIGN REVIEW PRIORITY LEVELS (VAL)")
print("="*80)
print("Priority = review urgency based on anomaly rank, NOT clinical risk prediction\n")

# Simple priority assignment based on anomaly score percentiles
# Priority 1: highest anomaly scores (most unusual)
# Priority 4: lowest anomaly scores among flagged (least unusual but still atypical)

n_flagged_val = len(val_flagged_indices)
priority_bins = np.array_split(np.arange(n_flagged_val), N_PRIORITY_LEVELS)

val_priorities = np.zeros(n_flagged_val, dtype=int)
for priority_level, bin_indices in enumerate(priority_bins, start=1):
    val_priorities[bin_indices] = priority_level

# Count distribution
val_priority_counts = {f"Priority_{i}": (val_priorities == i).sum() for i in range(1, N_PRIORITY_LEVELS+1)}

print(f"Review Priority distribution (VAL flagged, n={n_flagged_val}):")
for priority, count in val_priority_counts.items():
    pct = count / n_flagged_val * 100
    priority_num = int(priority.split('_')[1])
    if priority_num == 1:
        desc = "(most unusual - highest review priority)"
    elif priority_num == N_PRIORITY_LEVELS:
        desc = "(least unusual among flagged - lower review priority)"
    else:
        desc = "(moderate priority)"
    print(f"  {priority}: {count} ({pct:.1f}%) {desc}")

print("\nNote: Priority levels indicate review order, NOT clinical severity or treatment urgency.")
print("="*80)


STAGE 2B: ASSIGN REVIEW PRIORITY LEVELS (VAL)
Priority = review urgency based on anomaly rank, NOT clinical risk prediction

Review Priority distribution (VAL flagged, n=175):
  Priority_1: 44 (25.1%) (most unusual - highest review priority)
  Priority_2: 44 (25.1%) (moderate priority)
  Priority_3: 44 (25.1%) (moderate priority)
  Priority_4: 43 (24.6%) (least unusual among flagged - lower review priority)

Note: Priority levels indicate review order, NOT clinical severity or treatment urgency.


In [11]:
# CELL 11: STAGE 3 - DISCOVERY CLUSTERING (VAL FLAGGED) - GRACEFUL FALLBACK

print("\n" + "="*80)
print("STAGE 3: DISCOVERY CLUSTERING (VAL FLAGGED)")
print("="*80)

# Get flagged embeddings
val_flagged_embeddings = val_screen_embeddings[val_flagged_indices]

# Check if enough samples for clustering
if n_flagged_val < CLUSTER_MIN_SAMPLES:
    print(f"⚠️  Too few flagged cases for clustering (n={n_flagged_val} < {CLUSTER_MIN_SAMPLES} minimum)")
    print(f"    Assigning all cases to Cluster 0 (no subgroup discovery)\n")
    best_k_val = 1
    val_cluster_labels = np.zeros(n_flagged_val, dtype=int)
    silhouette_scores_val = {}
else:
    # Determine optimal k using silhouette score
    silhouette_scores_val = {}
    max_k_val = min(CLUSTER_MAX_K, n_flagged_val - 1)
    
    print(f"Testing k from {CLUSTER_MIN_K} to {max_k_val}...")
    
    for k in range(CLUSTER_MIN_K, max_k_val + 1):
        kmeans = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
        labels = kmeans.fit_predict(val_flagged_embeddings)
        score = silhouette_score(val_flagged_embeddings, labels)
        silhouette_scores_val[k] = score
        print(f"  k={k}: silhouette={score:.4f}")
    
    best_k_val = max(silhouette_scores_val, key=silhouette_scores_val.get)
    print(f"\n✓ Optimal k: {best_k_val} (silhouette={silhouette_scores_val[best_k_val]:.4f})")
    
    # Final clustering
    kmeans_val_final = KMeans(n_clusters=best_k_val, random_state=RANDOM_STATE, n_init=20)
    val_cluster_labels = kmeans_val_final.fit_predict(val_flagged_embeddings)

# Cluster statistics
print(f"\nCluster breakdown:")
for c in range(best_k_val):
    mask = (val_cluster_labels == c)
    size = mask.sum()
    
    cluster_scores = val_anomaly_scores[val_flagged_indices[mask]]
    cluster_preds = val_screen_predictions[val_flagged_indices[mask]]
    
    pred_dist = np.bincount(cluster_preds, minlength=4)
    
    print(f"\n  Cluster {c}: {size} cases ({size/n_flagged_val*100:.1f}%)")
    print(f"    Mean anomaly score: {cluster_scores.mean():.4f}")
    print(f"    Prediction dist: CNV={pred_dist[0]}, DME={pred_dist[1]}, DRUSEN={pred_dist[2]}, NORMAL={pred_dist[3]}")

print("\n" + "="*80)


STAGE 3: DISCOVERY CLUSTERING (VAL FLAGGED)
Testing k from 2 to 6...
  k=2: silhouette=0.4031
  k=3: silhouette=0.4455
  k=4: silhouette=0.4038
  k=5: silhouette=0.3703
  k=6: silhouette=0.3388

✓ Optimal k: 3 (silhouette=0.4455)

Cluster breakdown:

  Cluster 0: 35 cases (20.0%)
    Mean anomaly score: 40.3761
    Prediction dist: CNV=15, DME=16, DRUSEN=4, NORMAL=0

  Cluster 1: 93 cases (53.1%)
    Mean anomaly score: 17.7694
    Prediction dist: CNV=1, DME=12, DRUSEN=3, NORMAL=77

  Cluster 2: 47 cases (26.9%)
    Mean anomaly score: 37.3796
    Prediction dist: CNV=22, DME=0, DRUSEN=25, NORMAL=0



In [12]:
# CELL 12: APPLY TO TEST SET (ALL STAGES)

print("\n" + "="*80)
print("APPLYING TO TEST SET (HELD-OUT EVALUATION)")
print("="*80)

# Load TEST data
test_embeddings = np.load(TEST_EMBEDDINGS)
test_labels = np.load(TEST_LABELS)
test_predictions = np.load(TEST_PREDICTIONS)
test_probabilities = np.load(TEST_PROBABILITIES)

print(f"Total TEST samples: {len(test_embeddings):,}\n")

# Stage 0: Define screening population (same logic as VAL)
print("Stage 0: Screening...")
test_screen_mask, test_components = compute_screening_mask(
    test_predictions,
    test_probabilities,
    mode=SCREENING_MODE,
    margin_thresh=MARGIN_THRESHOLD,
    entropy_thresh=ENTROPY_THRESHOLD,
    disease_mass_thresh=DISEASE_MASS_THRESHOLD,
    
    
    normal_idx=NORMAL_CLASS_IDX
)



# Print detailed breakdown
print_screening_breakdown(test_components, SCREENING_MODE, len(test_embeddings))

# Extract screened population
test_screen_embeddings = test_embeddings[test_screen_mask]
test_screen_probabilities = test_probabilities[test_screen_mask]
test_screen_predictions = test_predictions[test_screen_mask]
test_screen_labels = test_labels[test_screen_mask]
test_screen_indices = np.where(test_screen_mask)[0]

print(f"  Screened: {len(test_screen_embeddings):,} / {len(test_embeddings):,} ({len(test_screen_embeddings)/len(test_embeddings)*100:.1f}%)")

# Stage 1: Compute anomaly scores
print("\nStage 1: Anomaly scoring...")
test_anomaly_scores = np.array([
    compute_mahalanobis(emb) for emb in test_screen_embeddings
])
print(f"  Scores: min={test_anomaly_scores.min():.4f}, max={test_anomaly_scores.max():.4f}, mean={test_anomaly_scores.mean():.4f}")

# Stage 1b: Enforce referral budget
print("\nStage 1B: Enforcing referral budget...")
test_flagged_indices, test_n_flagged, test_actual_rate = apply_referral_budget(
    test_anomaly_scores,
    budget=REFERRAL_BUDGET,
    rounding=BUDGET_ROUNDING
)

test_threshold = test_anomaly_scores[test_flagged_indices[-1]]

print(f"  Target: {REFERRAL_BUDGET * len(test_screen_embeddings):.1f} cases")
print(f"  Flagged: {test_n_flagged} ({test_actual_rate:.2%})")
print(f"  Effective threshold: {test_threshold:.4f}")

# Stage 2: Assign priorities
print("\nStage 2: Assigning review priorities...")
n_flagged_test = len(test_flagged_indices)
priority_bins_test = np.array_split(np.arange(n_flagged_test), N_PRIORITY_LEVELS)

test_priorities = np.zeros(n_flagged_test, dtype=int)
for priority_level, bin_indices in enumerate(priority_bins_test, start=1):
    test_priorities[bin_indices] = priority_level

test_priority_counts = {f"Priority_{i}": (test_priorities == i).sum() for i in range(1, N_PRIORITY_LEVELS+1)}
print(f"  Priority distribution: {test_priority_counts}")

# Stage 3: Cluster flagged cases
print("\nStage 3: Discovery clustering...")
test_flagged_embeddings = test_screen_embeddings[test_flagged_indices]

if n_flagged_test < CLUSTER_MIN_SAMPLES:
    print(f"  Too few cases (n={n_flagged_test} < {CLUSTER_MIN_SAMPLES}), assigning Cluster 0")
    best_k_test = 1
    test_cluster_labels = np.zeros(n_flagged_test, dtype=int)
else:
    # Use same k as VAL for consistency
    best_k_test = min(best_k_val, n_flagged_test - 1)
    if best_k_test >= 2:
        kmeans_test = KMeans(n_clusters=best_k_test, random_state=RANDOM_STATE, n_init=20)
        test_cluster_labels = kmeans_test.fit_predict(test_flagged_embeddings)
        print(f"  Clusters: k={best_k_test}")
        for c in range(best_k_test):
            count = (test_cluster_labels == c).sum()
            print(f"    Cluster {c}: {count} cases")
    else:
        test_cluster_labels = np.zeros(n_flagged_test, dtype=int)
        print(f"  Insufficient samples for k={best_k_val}, using Cluster 0")

print("\n" + "="*80)


APPLYING TO TEST SET (HELD-OUT EVALUATION)
Total TEST samples: 968

Stage 0: Screening...

Screening mode: MODE_B
Total samples: 968

COMPONENT BREAKDOWN
Primary components:
  • Predicted NORMAL: 241
  • Low margin (low confidence): 9
  • High entropy (high uncertainty): 0
  • Suspicious NORMAL (high disease_mass): 0

------------------------------------------------------------
Uncertainty analysis:
  • Total uncertain (margin OR entropy): 9

------------------------------------------------------------
Overlap analysis (how components combine):
  • NORMAL only (not uncertain): 241
  • Uncertain only (not NORMAL pred): 9
  • Both NORMAL and uncertain: 0

→ SCREENED (union): 250 (25.8% of total)

Interpretation:
  ℹ️  Mode B primarily screening NORMAL predictions (96%)
      Uncertainty criteria add relatively few cases
  Screened: 250 / 968 (25.8%)

Stage 1: Anomaly scoring...
  Scores: min=7.1362, max=49.7435, mean=10.5018

Stage 1B: Enforcing referral budget...
  Target: 12.5 cases
 

In [13]:
# CELL 13: USEFULNESS CHECK - ERROR ENRICHMENT ANALYSIS

print("\n" + "="*80)
print("USEFULNESS CHECK - ERROR ENRICHMENT ANALYSIS")
print("="*80)
print("Does the system flag cases with higher error rates?\n")

# Compute errors in flagged vs not-flagged
test_flagged_preds = test_screen_predictions[test_flagged_indices]
test_flagged_labels = test_screen_labels[test_flagged_indices]
test_flagged_errors = (test_flagged_preds != test_flagged_labels)

test_not_flagged_indices = np.array([i for i in range(len(test_screen_embeddings)) 
                                     if i not in test_flagged_indices])
test_not_flagged_preds = test_screen_predictions[test_not_flagged_indices]
test_not_flagged_labels = test_screen_labels[test_not_flagged_indices]
test_not_flagged_errors = (test_not_flagged_preds != test_not_flagged_labels)

# Error rates
n_errors_flagged = test_flagged_errors.sum()
n_errors_not_flagged = test_not_flagged_errors.sum()

error_rate_flagged = n_errors_flagged / len(test_flagged_indices) * 100 if len(test_flagged_indices) > 0 else 0
error_rate_not_flagged = n_errors_not_flagged / len(test_not_flagged_indices) * 100 if len(test_not_flagged_indices) > 0 else 0

# Enrichment factor
enrichment = error_rate_flagged / error_rate_not_flagged if error_rate_not_flagged > 0 else float('inf')

print(f"Error rates in screened TEST population:")
print(f"  Flagged cases (n={len(test_flagged_indices)}): {n_errors_flagged} errors ({error_rate_flagged:.1f}%)")
print(f"  Not flagged (n={len(test_not_flagged_indices)}): {n_errors_not_flagged} errors ({error_rate_not_flagged:.1f}%)")
print(f"\nError enrichment factor: {enrichment:.2f}x")

if enrichment > 1.5:
    print("\n✓ USEFUL: Flagged set is enriched for model errors (>1.5x)")
    print("  The system successfully identifies cases where the model is more likely wrong.")
elif enrichment > 1.0:
    print("\n⊙ MODEST: Flagged set has slightly higher error rate")
    print("  The system shows some ability to identify problematic cases.")
elif enrichment == 1.0:
    print("\n⊘ NEUTRAL: Error rate same in flagged vs not-flagged")
    print("  The system flags atypical patterns but not necessarily errors.")
else:
    print("\n⚠️  INVERSE: Flagged set has LOWER error rate than not-flagged")
    print("  This is unexpected - system may be flagging correct but unusual cases.")

# Break down by error type if any exist
if n_errors_flagged > 0:
    print(f"\nFlagged error breakdown (n={n_errors_flagged}):")
    error_indices = np.where(test_flagged_errors)[0]
    for idx in error_indices:
        pred_name = CLASS_NAMES[test_flagged_preds[idx]]
        true_name = CLASS_NAMES[test_flagged_labels[idx]]
        print(f"  Predicted {pred_name} → Actually {true_name}")

print("="*80)


USEFULNESS CHECK - ERROR ENRICHMENT ANALYSIS
Does the system flag cases with higher error rates?

Error rates in screened TEST population:
  Flagged cases (n=13): 5 errors (38.5%)
  Not flagged (n=237): 0 errors (0.0%)

Error enrichment factor: infx

✓ USEFUL: Flagged set is enriched for model errors (>1.5x)
  The system successfully identifies cases where the model is more likely wrong.

Flagged error breakdown (n=5):
  Predicted CNV → Actually DRUSEN
  Predicted CNV → Actually DRUSEN
  Predicted CNV → Actually DRUSEN
  Predicted CNV → Actually DRUSEN
  Predicted DRUSEN → Actually NORMAL


In [14]:
# CELL 14: GENERATE RANKED RESULTS TABLE (INTERPRETABILITY)

print("\n" + "="*80)
print("RANKED RESULTS TABLE (TEST FLAGGED CASES)")
print("="*80)

# Extract all information for flagged cases
test_flagged_probs = test_screen_probabilities[test_flagged_indices]

# Compute explanation signals
sorted_probs_test = np.sort(test_flagged_probs, axis=1)[:, ::-1]
test_flagged_margin = sorted_probs_test[:, 0] - sorted_probs_test[:, 1]

epsilon = 1e-10
test_flagged_entropy = -np.sum(test_flagged_probs * np.log(test_flagged_probs + epsilon), axis=1)
test_flagged_disease_mass = test_flagged_probs[:, 0] + test_flagged_probs[:, 1] + test_flagged_probs[:, 2]

# Build results dataframe
results_data = []
for i, idx in enumerate(test_flagged_indices):
    original_test_idx = test_screen_indices[idx]
    
    results_data.append({
        'Rank': i + 1,
        'Test_Index': int(original_test_idx),
        'Predicted': CLASS_NAMES[test_screen_predictions[idx]],
        'True_Label': CLASS_NAMES[test_screen_labels[idx]],
        'Is_Error': 'YES' if test_screen_predictions[idx] != test_screen_labels[idx] else 'NO',
        'Anomaly_Score': float(test_anomaly_scores[idx]),
        'Margin': float(test_flagged_margin[i]),
        'Entropy': float(test_flagged_entropy[i]),
        'Disease_Mass': float(test_flagged_disease_mass[i]),
        'Priority': int(test_priorities[i]),
        'Cluster': int(test_cluster_labels[i]) if best_k_test >= 2 else 0
    })

results_df = pd.DataFrame(results_data)

# Display table
print("\nComplete ranked table (sorted by anomaly score, highest first):\n")
pd.set_option('display.max_rows', None)
pd.set_option('display.width', None)
pd.set_option('display.max_columns', None)
print(results_df.to_string(index=False))

print("\n" + "="*80)

# Save to CSV for detailed review
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
ranked_csv = RESULTS_DIR / f"test_flagged_ranked_{timestamp}.csv"
results_df.to_csv(ranked_csv, index=False)
print(f"\n✓ Ranked table saved: {ranked_csv.name}")


RANKED RESULTS TABLE (TEST FLAGGED CASES)

Complete ranked table (sorted by anomaly score, highest first):

 Rank  Test_Index Predicted True_Label Is_Error  Anomaly_Score   Margin  Entropy  Disease_Mass  Priority  Cluster
    1         699       CNV     DRUSEN      YES      49.743486 0.205521 0.672567      1.000000         1        0
    2         618    DRUSEN     DRUSEN       NO      48.622940 0.044081 0.693978      0.999996         1        0
    3         253       DME        DME       NO      47.802708 0.265882 0.764858      0.999871         1        2
    4         595    DRUSEN     DRUSEN       NO      44.833479 0.074424 0.690398      1.000000         1        0
    5         624       CNV     DRUSEN      YES      42.294262 0.286326 0.651639      0.999998         2        0
    6         576       CNV     DRUSEN      YES      40.933845 0.162358 0.679940      0.999998         2        0
    7         551       CNV     DRUSEN      YES      37.765128 0.087963 0.689363      0.99999

In [15]:
# CELL 15: SAVE COMPREHENSIVE RESULTS (JSON)

print("\n" + "="*80)
print("SAVING COMPREHENSIVE RESULTS")
print("="*80)

# Compile all results
comprehensive_results = {
    "metadata": {
        "timestamp": datetime.now().isoformat(),
        "version": "final_integrated_version",
        "purpose": "quality_control_second_look_queue"
    },
    
    "configuration": {
        "screening_mode": SCREENING_MODE,
        "detection_method": DETECTION_METHOD,
        "referral_budget": REFERRAL_BUDGET,
        "budget_rounding": BUDGET_ROUNDING,
        "n_priority_levels": N_PRIORITY_LEVELS,
        "cluster_min_samples": CLUSTER_MIN_SAMPLES,
        "mode_b_thresholds": {
            "margin": MARGIN_THRESHOLD,
            "entropy": ENTROPY_THRESHOLD,
            "disease_mass_threshold": DISEASE_MASS_THRESHOLD  # FIXED: single threshold
        } if SCREENING_MODE == "mode_b" else None
    },
    
    "val_results": {
        "total_samples": int(len(val_embeddings)),
        "screened": int(len(val_screen_embeddings)),
        "screened_pct": float(len(val_screen_embeddings) / len(val_embeddings) * 100),
        "screening_components": {k: int(v) for k, v in val_components.items()},
        "flagged": int(val_n_flagged),
        "flagged_rate_target": float(REFERRAL_BUDGET * 100),
        "flagged_rate_actual": float(val_actual_rate * 100),
        "effective_threshold": float(val_threshold),
        "priority_counts": {k: int(v) for k, v in val_priority_counts.items()},
        "n_clusters": int(best_k_val),
        "cluster_sizes": [int((val_cluster_labels == c).sum()) for c in range(best_k_val)]
    },
    
    "test_results": {
        "total_samples": int(len(test_embeddings)),
        "screened": int(len(test_screen_embeddings)),
        "screened_pct": float(len(test_screen_embeddings) / len(test_embeddings) * 100),
        "screening_components": {k: int(v) for k, v in test_components.items()},
        "flagged": int(test_n_flagged),
        "flagged_rate_target": float(REFERRAL_BUDGET * 100),
        "flagged_rate_actual": float(test_actual_rate * 100),
        "effective_threshold": float(test_threshold),
        "priority_counts": {k: int(v) for k, v in test_priority_counts.items()},
        "n_clusters": int(best_k_test) if n_flagged_test >= CLUSTER_MIN_SAMPLES else 1,
        "cluster_sizes": [int((test_cluster_labels == c).sum()) for c in range(max(1, best_k_test))],
        "error_enrichment": {
            "flagged_errors": int(n_errors_flagged),
            "flagged_error_rate": float(error_rate_flagged),
            "not_flagged_errors": int(n_errors_not_flagged),
            "not_flagged_error_rate": float(error_rate_not_flagged),
            "enrichment_factor": float(enrichment)
        }
    },
    
    "sensitivity_analysis": sensitivity_results if RUN_SENSITIVITY_ANALYSIS and SCREENING_MODE == "mode_b" else None
}

# Save JSON
results_json = RESULTS_DIR / f"early_warning_final_{timestamp}.json"
with open(results_json, 'w') as f:
    json.dump(comprehensive_results, f, indent=2)

print(f"✓ Comprehensive results: {results_json.name}")
print(f"✓ Ranked table (CSV): {ranked_csv.name}")
print(f"\nAll files saved to: {RESULTS_DIR}")
print("="*80)


SAVING COMPREHENSIVE RESULTS
✓ Comprehensive results: early_warning_final_20260226_003900.json
✓ Ranked table (CSV): test_flagged_ranked_20260226_003900.csv

All files saved to: C:\Users\Ajant\Documents\Project MSc - CSC-40098\3 - Experiments\Early_Warning_Detection\Results_Final_Integrated


In [16]:
# CELL 16: SUMMARY REPORT

print("\n" + "="*80)
print("EARLY WARNING SYSTEM - SUMMARY")
print("="*80)

summary_lines = []

summary_lines.append("\n" + "="*80)
summary_lines.append("WHAT THIS SYSTEM DOES")
summary_lines.append("="*80)
summary_lines.append("This is a quality-control system that acts as a 'second look' for cases")
summary_lines.append("that might benefit from human review. It does NOT diagnose or predict disease.")
summary_lines.append("Instead, it identifies cases with unusual patterns that differ from typical")
summary_lines.append("healthy scans, even when the AI model predicts them as NORMAL.")

summary_lines.append("\n" + "="*80)
summary_lines.append("HOW IT WORKS")
summary_lines.append("="*80)

if SCREENING_MODE == "mode_a":
    summary_lines.append("Step 1: Screen only cases predicted as NORMAL by the AI model")
elif SCREENING_MODE == "mode_b":
    summary_lines.append("Step 1: Screen cases predicted as NORMAL plus uncertain/borderline cases")
    summary_lines.append("        (cases with low confidence, high uncertainty, or mixed signals)")

summary_lines.append("")
summary_lines.append("Step 2: Compare each case to what typical healthy scans look like using")
summary_lines.append("        deep learning features (embeddings), NOT probabilities")
summary_lines.append("")
summary_lines.append(f"Step 3: Flag the top {REFERRAL_BUDGET*100:.0f}% most unusual cases for human review")
summary_lines.append("        (this percentage is guaranteed - it's not a learned threshold)")
summary_lines.append("")
summary_lines.append("Step 4: Organize flagged cases into review priority levels (1=highest to 4=lowest)")
summary_lines.append("        based on how unusual they are, plus provide explanation signals")
summary_lines.append("")
summary_lines.append("Step 5: Discover subgroups among flagged cases using clustering to identify")
summary_lines.append("        different types of atypical patterns")

summary_lines.append("\n" + "="*80)
summary_lines.append("VALIDATION SET RESULTS (FITTING DATA)")
summary_lines.append("="*80)
summary_lines.append(f"Total validation samples: {len(val_embeddings):,}")
summary_lines.append(f"Screened for review: {len(val_screen_embeddings):,} ({len(val_screen_embeddings)/len(val_embeddings)*100:.1f}%)")
summary_lines.append(f"Flagged as atypical: {val_n_flagged} ({val_actual_rate*100:.1f}% of screened)")
summary_lines.append(f"Subgroups discovered: {best_k_val}")

summary_lines.append("\n" + "="*80)
summary_lines.append("TEST SET RESULTS (HELD-OUT EVALUATION - MOST IMPORTANT)")
summary_lines.append("="*80)
summary_lines.append(f"Total test samples: {len(test_embeddings):,}")
summary_lines.append(f"Screened for review: {len(test_screen_embeddings):,} ({len(test_screen_embeddings)/len(test_embeddings)*100:.1f}%)")
summary_lines.append(f"Flagged as atypical: {test_n_flagged} ({test_actual_rate*100:.1f}% of screened)")

summary_lines.append("\nWhat was flagged?")
if test_n_flagged > 0:
    summary_lines.append(f"  - {test_n_flagged} cases identified for human review")
    summary_lines.append(f"  - {n_errors_flagged} of these were actual model errors ({error_rate_flagged:.1f}%)")
    summary_lines.append(f"  - Organized into {N_PRIORITY_LEVELS} review priority levels")
    if best_k_test >= 2:
        summary_lines.append(f"  - Grouped into {best_k_test} distinct clusters (subgroups)")

summary_lines.append("\nDid the system find useful cases?")
if enrichment > 1.5:
    summary_lines.append(f"  YES - Flagged cases have {enrichment:.1f}x higher error rate than not-flagged cases")
    summary_lines.append(f"  The system successfully identifies cases where the AI is more likely wrong.")
elif enrichment > 1.0:
    summary_lines.append(f"  SOMEWHAT - Flagged cases have {enrichment:.1f}x higher error rate")
    summary_lines.append(f"  The system shows some ability to catch problematic cases.")
elif n_errors_flagged > 0:
    summary_lines.append(f"  NEUTRAL - The system caught {n_errors_flagged} error(s) but also flags unusual-but-correct cases")
    summary_lines.append(f"  This suggests the system identifies atypical patterns, not just errors.")
else:
    summary_lines.append(f"  NO ERRORS CAUGHT - All flagged cases were correctly predicted by the model")
    summary_lines.append(f"  However, these cases still showed unusual patterns worth investigating.")

summary_lines.append("\n" + "="*80)
summary_lines.append("REVIEW PRIORITY DISTRIBUTION (TEST)")
summary_lines.append("="*80)
summary_lines.append("Priority indicates review urgency based on pattern unusualness:")
for i in range(1, N_PRIORITY_LEVELS + 1):
    count = test_priority_counts[f'Priority_{i}']
    if i == 1:
        desc = "(Most unusual - review first)"
    elif i == N_PRIORITY_LEVELS:
        desc = "(Least unusual among flagged)"
    else:
        desc = "(Moderate priority)"
    summary_lines.append(f"  Priority {i}: {count} cases {desc}")

if best_k_test >= 2:
    summary_lines.append("\n" + "="*80)
    summary_lines.append("DISCOVERED SUBGROUPS (TEST)")
    summary_lines.append("="*80)
    summary_lines.append(f"The system found {best_k_test} distinct types of atypical patterns:")
    for c in range(best_k_test):
        count = (test_cluster_labels == c).sum()
        summary_lines.append(f"  Subgroup {c}: {count} cases ({count/test_n_flagged*100:.1f}%)")
    summary_lines.append("\n(See detailed table for specific cases in each subgroup)")

summary_lines.append("\n" + "="*80)
summary_lines.append("WHAT THIS MEANS")
summary_lines.append("="*80)
summary_lines.append(f"The system successfully identified {test_n_flagged} cases out of {len(test_screen_embeddings):,} screened")
summary_lines.append(f"that show unusual patterns compared to typical healthy scans. These cases")
summary_lines.append(f"represent a focused set for human expert review.")
summary_lines.append("")
summary_lines.append("Key strengths:")
summary_lines.append("  - Guarantees exact referral budget (no threshold guessing)")
summary_lines.append("  - Uses deep features (embeddings) not just model confidence")
summary_lines.append("  - Provides interpretable priority levels and subgroup discovery")
summary_lines.append("  - Focuses expert attention on genuinely atypical cases")
summary_lines.append("")
summary_lines.append("This is a quality-control tool, NOT a clinical decision system.")
summary_lines.append("All flagged cases should be reviewed by qualified experts.")

summary_lines.append("\n" + "="*80)
summary_lines.append("FILES GENERATED")
summary_lines.append("="*80)
summary_lines.append(f"1. Comprehensive results (JSON): {results_json.name}")
summary_lines.append(f"2. Ranked flagged cases (CSV): {ranked_csv.name}")
summary_lines.append(f"   (Contains detailed information for all {test_n_flagged} flagged cases)")
summary_lines.append(f"\nAll files saved to: {RESULTS_DIR}")

summary_lines.append("\n" + "="*80)
summary_lines.append("END OF SUMMARY")
summary_lines.append("="*80)

# Print summary
for line in summary_lines:
    print(line)

# Save summary to text file
summary_txt = RESULTS_DIR / f"plain_english_summary_{timestamp}.txt"
with open(summary_txt, 'w') as f:
    f.write('\n'.join(summary_lines))

print(f"\n✓ Plain English summary saved: {summary_txt.name}")
print("\n✅ EARLY WARNING SYSTEM COMPLETE")


EARLY WARNING SYSTEM - SUMMARY

WHAT THIS SYSTEM DOES
This is a quality-control system that acts as a 'second look' for cases
that might benefit from human review. It does NOT diagnose or predict disease.
Instead, it identifies cases with unusual patterns that differ from typical
healthy scans, even when the AI model predicts them as NORMAL.

HOW IT WORKS
Step 1: Screen cases predicted as NORMAL plus uncertain/borderline cases
        (cases with low confidence, high uncertainty, or mixed signals)

Step 2: Compare each case to what typical healthy scans look like using
        deep learning features (embeddings), NOT probabilities

Step 3: Flag the top 5% most unusual cases for human review
        (this percentage is guaranteed - it's not a learned threshold)

Step 4: Organize flagged cases into review priority levels (1=highest to 4=lowest)
        based on how unusual they are, plus provide explanation signals

Step 5: Discover subgroups among flagged cases using clustering to iden

In [17]:
# CELL 17: CLINICAL-STYLE SUMMARY REPORT (SIMPLE FORMAT)

print("\n" + "="*80)
print("EARLY WARNING SYSTEM - CLINICAL-STYLE SUMMARY REPORT")
print("="*80)

print("\n" + "─"*80)
print("OVERVIEW")
print("─"*80)
print(f"Dataset: TEST (Held-out evaluation)")
print(f"Total cases evaluated: {len(test_embeddings):,}")
print(f"Cases screened for review: {len(test_screen_embeddings):,} ({len(test_screen_embeddings)/len(test_embeddings)*100:.1f}%)")
print(f"Cases flagged for attention: {test_n_flagged} ({test_actual_rate*100:.1f}% of screened)")
print(f"Distinct pattern groups: {best_k_test}")

print("\n" + "─"*80)
print("🎯 FLAGGED CASES BY REVIEW PRIORITY")
print("─"*80)
print("Priority levels indicate urgency of review based on pattern unusualness.\n")

# Map priority to clinical-style presentation
priority_labels = {
    1: {"emoji": "🔴", "label": "URGENT", "action": "Immediate expert review within 1 week"},
    2: {"emoji": "🟠", "label": "HIGH", "action": "Expert review within 2-3 weeks"},
    3: {"emoji": "🟡", "label": "MODERATE", "action": "Scheduled review within 1 month"},
    4: {"emoji": "🔵", "label": "ROUTINE", "action": "Standard review queue"}
}

print(f"{'Priority':<12} {'Level':<12} {'Count':<8} {'Recommended Action'}")
print("─"*80)

for priority in range(1, N_PRIORITY_LEVELS + 1):
    info = priority_labels[priority]
    count = test_priority_counts[f'Priority_{priority}']
    print(f"{info['emoji']} Priority {priority}  {info['label']:<12} {count:<8} {info['action']}")

print("\n" + "─"*80)
print("DISCOVERED PATTERN GROUPS (CLUSTERS)")
print("─"*80)
print("Flagged cases grouped by similarity in atypical patterns.\n")

print(f"{'Group':<8} {'Size':<8} {'% of Flagged':<15} {'Characteristics'}")
print("─"*80)

for cluster_id in range(best_k_test):
    cluster_mask = (test_cluster_labels == cluster_id)
    cluster_size = cluster_mask.sum()
    cluster_pct = cluster_size / test_n_flagged * 100
    
    # Get some characteristics
    cluster_indices = test_flagged_indices[cluster_mask]
    cluster_preds = test_screen_predictions[cluster_indices]
    cluster_labels_true = test_screen_labels[cluster_indices]
    cluster_scores = test_anomaly_scores[cluster_indices]
    
    # Count predictions and errors
    pred_dist = np.bincount(cluster_preds, minlength=4)
    n_errors = (cluster_preds != cluster_labels_true).sum()
    
    # Build characteristic string
    characteristics = []
    if n_errors > 0:
        characteristics.append(f"{n_errors} model error(s)")
    characteristics.append(f"Avg anomaly: {cluster_scores.mean():.1f}")
    
    # Most common prediction
    most_common_pred = np.argmax(pred_dist)
    characteristics.append(f"Mostly predicted {CLASS_NAMES[most_common_pred]}")
    
    char_str = " | ".join(characteristics)
    
    print(f"Group {cluster_id}   {cluster_size:<8} {cluster_pct:>6.1f}%        {char_str}")

print("\n" + "─"*80)
print("⚠️  ERROR DETECTION PERFORMANCE")
print("─"*80)
print("How well does the system catch model mistakes?\n")

print(f"{'Metric':<40} {'Value'}")
print("─"*80)
print(f"{'Total model errors in screened population':<40} {n_errors_flagged + n_errors_not_flagged}")
print(f"{'Errors caught by system (flagged)':<40} {n_errors_flagged} ({error_rate_flagged:.1f}%)")
print(f"{'Errors missed (not flagged)':<40} {n_errors_not_flagged} ({error_rate_not_flagged:.1f}%)")
print(f"{'Error enrichment factor':<40} {enrichment:.2f}x")

print("\n")
if enrichment > 1.5:
    print("✅ EXCELLENT: System strongly enriches for model errors")
elif enrichment > 1.0:
    print("✓ GOOD: System identifies some problematic cases")
elif enrichment == 1.0:
    print("○ NEUTRAL: System finds atypical patterns, not specifically errors")
else:
    print("⚠ UNEXPECTED: Flagged cases have lower error rate")

print("\n" + "─"*80)
print("DETAILED CASE LIST")
print("─"*80)
print(f"See CSV file for complete ranked list of all {test_n_flagged} flagged cases:")
print(f"  → {ranked_csv.name}")
print(f"\nEach case includes:")
print(f"  • Test index (for lookup)")
print(f"  • Model prediction vs true label")
print(f"  • Error flag (YES/NO)")
print(f"  • Anomaly score")
print(f"  • Confidence metrics (margin, entropy, disease_mass)")
print(f"  • Priority level (1-4)")
print(f"  • Pattern group (0-{best_k_test-1})")

print("\n" + "─"*80)
print("OUTPUT FILES LOCATION")
print("─"*80)
print(f"All results saved to:")
print(f"  {RESULTS_DIR}")
print(f"\nFiles generated:")
print(f"  1. {results_json.name}")
print(f"     (Complete technical results in JSON format)")
print(f"  2. {ranked_csv.name}")
print(f"     (Detailed table - open in Excel)")
print(f"  3. {summary_txt.name}")
print(f"     (This report in text format)")

print("\n" + "="*80)
print("✅ EARLY WARNING SYSTEM - REPORT COMPLETE")
print("="*80)

print("\nNEXT STEPS:")
print("  1. Review the ranked CSV file to see individual flagged cases")
print("  2. Start with Priority 1 cases (most unusual patterns)")
print("  3. Investigate why each case was flagged (check anomaly score, signals)")
print("  4. Group-level review: examine each pattern group for common features")
print("  5. Expert validation: have specialists review flagged cases")

print("\n⚠️  IMPORTANT REMINDERS:")
print("  • This is a quality-control tool, NOT a diagnostic system")
print("  • All flagged cases require expert human review")
print("  • Priority levels indicate review urgency, not clinical severity")
print("  • Pattern groups suggest investigation starting points")
print("  • The system identifies unusual patterns, not treatment recommendations")

print("\n" + "="*80)


EARLY WARNING SYSTEM - CLINICAL-STYLE SUMMARY REPORT

────────────────────────────────────────────────────────────────────────────────
OVERVIEW
────────────────────────────────────────────────────────────────────────────────
Dataset: TEST (Held-out evaluation)
Total cases evaluated: 968
Cases screened for review: 250 (25.8%)
Cases flagged for attention: 13 (5.2% of screened)
Distinct pattern groups: 3

────────────────────────────────────────────────────────────────────────────────
🎯 FLAGGED CASES BY REVIEW PRIORITY
────────────────────────────────────────────────────────────────────────────────
Priority levels indicate urgency of review based on pattern unusualness.

Priority     Level        Count    Recommended Action
────────────────────────────────────────────────────────────────────────────────
🔴 Priority 1  URGENT       4        Immediate expert review within 1 week
🟠 Priority 2  HIGH         3        Expert review within 2-3 weeks
🟡 Priority 3  MODERATE     3        Scheduled 